In [ ]:
dataset

In [7]:
!pip install tqdm fastparquet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 722.7/722.7 kB 19.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 58.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [fastparquet]


In [6]:
import requests
import yaml
import getpass
# import zstandard as zstd
import pandas as pd
import json
import io

from typing import Dict, List, Any
from tqdm import tqdm

In [31]:
df_queries = pd.read_parquet('esci-data/shopping_queries_dataset/shopping_queries_dataset_examples.parquet')
df_queries = df_queries[df_queries["product_locale"] == "us"]


In [32]:
df_queries.head(5)

,example_id,query,query_id,product_id,product_locale,esci_label,small_version,large_version,split
0,0,revent 80 cfm,0,B000MOO21W,us,I,0,1,train
1,1,revent 80 cfm,0,B07X3Y6B1V,us,E,0,1,train
2,2,revent 80 cfm,0,B07WDM7MQQ,us,E,0,1,train
3,3,revent 80 cfm,0,B07RH6Z8KW,us,E,0,1,train
4,4,revent 80 cfm,0,B07QJ7WYFQ,us,E,0,1,train


In [37]:
unique_queries = df_queries["query"].drop_duplicates()

In [38]:
len(unique_queries)

97345

In [39]:
unique_queries.head(5)

0                                revent 80 cfm
16                !awnmower tires without rims
32                !qscreen fence without holes
149    # 10 self-seal envelopes without window
189                  # 2 pencils not sharpened
Name: query, dtype: object

In [40]:
random_queries = unique_queries.sample(n=1000, random_state=42)
len(random_queries)

1000

In [27]:
df_products = pd.read_parquet('esci-data/shopping_queries_dataset/shopping_queries_dataset_products.parquet')
df_products = df_products[df_products["product_locale"] == "us"]
df_products.head(5)

,product_id,product_title,product_description,product_bullet_point,product_brand,product_color,product_locale
167168,B003O0MNGC,Delta BreezSignature VFB25ACH 80 CFM Exhaust B...,None,Virtually silent at less than 0.3 sones\nPreci...,DELTA ELECTRONICS (AMERICAS) LTD.,White,us
167169,B00MARNO5Y,Aero Pure AP80RVLW Super Quiet 80 CFM Recessed...,None,Super quiet 80CFM energy efficient fan virtual...,Aero Pure,White,us
167170,B011RX6PNO,Aero Pure AP120H-SL W Slim Fit 120 CFM Bathroo...,None,"Slim Fit Housing Fits Into 2"" X 6"" Ceiling Joi...",Aero Pure,White Finish,us
167171,B01MZIK0PI,Delta Electronics (Americas) Ltd. RAD80 Delta ...,None,Quiet operation at 1.5 Sones\nPrecision engine...,DELTA ELECTRONICS (AMERICAS) LTD.,With Heater,us
167172,B01N5Y6002,Delta Electronics (Americas) Ltd. GBR80HLED De...,None,Ultra energy-efficient LED module (11-watt equ...,DELTA ELECTRONICS (AMERICAS) LTD.,"With LED Light, Dual Speed & Humidity Sensor",us


In [42]:
df_random_queries = df_queries[
    df_queries["query"].isin(random_queries)
]

In [43]:
df = pd.merge(
    df_random_queries,
    df_products,
    how='left',
    left_on=['product_locale','product_id'],
    right_on=['product_locale', 'product_id']
)
df.head(5)

,example_id,query,query_id,product_id,product_locale,esci_label,small_version,large_version,split,product_title,product_description,product_bullet_point,product_brand,product_color
0,3042,$5 items,100,B079HXXP4T,us,I,1,1,test,"Soft Scrub In-Tank Toilet Cleaner Duo-Cubes, A...",None,"Helps fight toilet ring, hard water, and limes...",Soft Scrub,Alpine Fresh
1,3043,$5 items,100,B07DZYGDS3,us,E,1,1,test,"Gillette Fusion5 Razors for Men, 1 Gillette Ra...",None,REFILLS FIT ALL GILLETTE 5-BLADE RAZOR HANDLES...,Gillette,None
2,3044,$5 items,100,B07HY9DC4N,us,E,1,1,test,"Summer's Eve Cleansing Cloths, Blissful Escape...",None,Summer's Eve Feminine Cleansing Wipes are safe...,Summer's Eve,None
3,3045,$5 items,100,B07M77RB97,us,E,1,1,test,BIC Flex 5 Hybrid Men's 5-Blade Disposable Raz...,None,"5 long lasting, flexible blades individually a...",BIC,Black
4,3046,$5 items,100,B07NTWYGJX,us,I,1,1,test,"6PCS Dual Heads Blackhead Remover, Pimple Come...",<b>About Our Factory:</b><br /> ✿✿Our factory ...,"♥ 【Dual Heads REMOVER】: 6PCS dual heads tools,...",USCOLOR,None


In [50]:
len(df)

18727

In [ ]:
index

In [45]:
SEARCH_INDEX = 'http://localhost:9200/myindex'

In [46]:
idx = requests.put(
    SEARCH_INDEX,
    json={
        "mappings": {
            "properties": {
                "name": {
                    "type": "text"
                },
                "description": {
                    "type": "text"
                }
            }
        }
    }
)
idx.json()

{'acknowledged': True, 'shards_acknowledged': True, 'index': 'myindex'}

In [47]:
def index_record(id, name, description):
    if id and name and description:
        try:
            return requests.post(
                f"{SEARCH_INDEX}/_doc/{id}",
                json={
                    'name': name,
                    'description': description    
                }
            )
        except:
            pass

In [48]:
for index, row in tqdm(df.iterrows(), total=len(df)):
    _ = index_record(row['example_id'], row['product_title'], row['product_description'])

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████| 18727/18727 [00:23<00:00, 808.38it/s]


In [49]:
response = requests.post(
    f"{SEARCH_INDEX}/_search",
    json={
        "size": 0,
        "track_total_hits": True
    }
)
response.json()

{'took': 256,
 'timed_out': False,
 'terminated_early': False,
 '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0},
 'hits': {'total': {'value': 9295, 'relation': 'eq'},
  'max_score': None,
  'hits': []}}

In [52]:
def search_query(query='#$query##'):
    return {
      "query": {
        "multi_match": {
          "query": query,
          "fields": [f"name", "description"]
        }
      }
    }
    
def search(query):
    response = requests.post(
        f"{SEARCH_INDEX}/_search",
        json=search_query(query)
    )
    return response.json()

In [53]:
search('dinosaur')

{'took': 49,
 'timed_out': False,
 '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0},
 'hits': {'total': {'value': 31, 'relation': 'eq'},
  'max_score': 13.017879,
  'hits': [{'_index': 'myindex',
    '_id': '1975317',
    '_score': 13.017879,
    '_source': {'name': 'Baby Dinosaur Balloon Set for Birthday Decor - 38 Inch, Pack of 4, 4D Dinosaur Foil Balloon | Kids Dinosaur Party Decorations | Dinosaur Balloons for Birthday Party | Dinosaur Birthday Party Supplies',
     'description': '<p><b>Are you a dinasaur fanatic?</b></p><p>Looking for a dinosaur theme party decorations for your kid</p> <p>We have got you covered with these beautiful and gigantic <b>Baby Dinosaur Foil Balloons </b> in two different style of dinosaur party balloons </p> <p>This dinosaur balloon kit is simply beautiful, gorgeous color to your dinosaur balloon birthday party as a backdrop for photos Booth.</p> <p>Configure it any way you wish; there are 1000s of ways to use your kit to build your b

## judge

In [ ]:
train student

In [ ]:
compare

In [ ]:
human labels - better teacher - dspy??

In [ ]:
https://github.com/amazon-science/esci-data/blob/main/shopping_queries_dataset/shopping_queries_dataset_products.parquet